# DeepSeek Architecture

The GPT architecture from [Notebook 01](/notebooks/llm/01-gpt-architecture.html) is correct and fully trainable, but it has two well-known inefficiencies at scale. First, the **KV cache** grows as $O(n_{\text{heads}} \cdot d_{\text{head}} \cdot \text{seq\_len})$ per layer — for a 27B model generating 8k tokens this can exceed the model weight memory. Second, the **dense FFN** activates every parameter for every token on every forward pass, spending equal compute regardless of what knowledge a token actually needs.

DeepSeek-V3 solves both with two architectural innovations. **Multi-head Latent Attention (MLA)** compresses keys and values through a low-rank bottleneck, caching only the bottleneck vector rather than the full K/V tensors. **Mixture of Experts (MoE)** replaces the dense FFN with a bank of expert networks, routing each token to only the top-$k$ experts — large total parameter count, small *activated* parameter count.

This notebook builds both from scratch and assembles them into `NanoDeepSeek`. We derive the low-rank KV compression and decoupled RoPE for MLA, implement top-$k$ routing with auxiliary load-balancing loss for MoE, then close with a side-by-side comparison of NanoGPT vs NanoDeepSeek on parameter count, KV cache footprint, and activated FLOPs.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple

## The KV Cache Bottleneck

During inference, autoregressive generation processes one token at a time. At step $t$ the model needs keys and values for all previous tokens $1, \ldots, t-1$. Recomputing them from scratch costs $O(t^2)$ total, so instead we cache them: after processing token $i$ we store $K_i$ and $V_i$ for every layer. This is the KV cache.

The memory cost per token per layer for standard MHA:

$$\text{KV cache per token per layer} = 2 \times n_{\text{heads}} \times d_{\text{head}} \times \text{bytes\_per\_element}$$

For the NanoGPT config (`n_heads=6`, `d_model=384`, so `d_head=64`, bfloat16 = 2 bytes):

$$2 \times 6 \times 64 \times 2 = 1{,}536 \text{ bytes per token per layer}$$

For a 6-layer model generating 256 tokens that is $1{,}536 \times 256 \times 6 = 2.36$ MB — manageable. But scale to 30 layers and 8k context and it grows to hundreds of MB, competing with the model weights themselves.

[MLA's insight: we don't need to cache the full $K$ and $V$ matrices.]{.mark} We can cache a single low-rank *latent vector* $c_{KV}$ per token per layer and reconstruct $K$ and $V$ from it on demand during attention. With latent dimension $d_c \ll n_{\text{heads}} \times d_{\text{head}}$, the cache shrinks dramatically.

In [ ]:
# KV cache memory comparison: standard MHA vs MLA
def kv_cache_bytes(n_heads, d_head, n_layers, seq_len, bytes_per_elem=2):
    """Standard MHA: cache K and V for all heads."""
    return 2 * n_heads * d_head * n_layers * seq_len * bytes_per_elem

def kv_cache_bytes_mla(d_compressed, n_layers, seq_len, bytes_per_elem=2):
    """MLA: cache only the low-rank latent c_KV per token."""
    return d_compressed * n_layers * seq_len * bytes_per_elem

# NanoGPT config
n_heads, d_model, n_layers = 6, 384, 6
d_head = d_model // n_heads   # 64

# NanoDeepSeek: d_compressed = d_model // 4
d_c = d_model // 4   # 96

for seq_len in [256, 1024, 4096]:
    mha = kv_cache_bytes(n_heads, d_head, n_layers, seq_len)
    mla = kv_cache_bytes_mla(d_c, n_layers, seq_len)
    print(f"seq_len={seq_len:5d} | MHA: {mha/1024:8.1f} KB | MLA: {mla/1024:6.1f} KB | ratio: {mha/mla:.1f}x")

## Multi-Head Latent Attention (MLA)

### Low-rank KV compression

Standard MHA projects the input $x \in \mathbb{R}^{d}$ directly into queries, keys, and values:

$$Q = x W_Q, \quad K = x W_K, \quad V = x W_V$$

MLA replaces the K and V projections with a two-step process. First, **compress** to a low-rank latent:

$$c_{KV} = x W_c \quad \in \mathbb{R}^{d_c}, \qquad d_c \ll n_{\text{heads}} \times d_{\text{head}}$$

Then **expand** from the latent back to full K and V:

$$K = c_{KV} W_K, \qquad V = c_{KV} W_V$$

At inference we cache only $c_{KV}$ — one vector of dimension $d_c$ per token per layer. When computing attention at step $t$ we reconstruct K and V for all cached positions by passing their stored $c_{KV}$ through $W_K$ and $W_V$. Queries are also compressed for efficiency, though this does not affect cache size since queries are never cached:

$$c_Q = x W_{cQ} \in \mathbb{R}^{d_{cQ}}, \qquad Q = c_Q W_Q$$

### Decoupled RoPE

There is a subtlety with rotary positional encoding (RoPE). RoPE applies a position-dependent rotation to Q and K before the dot product. But if we cache $c_{KV}$ and reconstruct K later, the rotated K must be computed at attention time — not at caching time. Rotating $K$ at caching time would embed the position into the cached latent, breaking reconstruction.

MLA solves this with **decoupled RoPE**: a separate $d_r$-dimensional slice of Q and K carries the rotary encoding. This slice is *not* part of the compressed latent — it is computed fresh from the original input $x$ via dedicated projections $W_{QR}$, $W_{KR}$ at attention time. The cache stores $c_{KV}$ plus a compact RoPE key slice $k_R$.

In the nano-scale implementation below we use a simplified version: single-path Q, no separate decoupled RoPE slice, with RoPE applied after reconstruction. The KV cache savings are identical.

In [ ]:
def apply_rope(x, cos, sin):
    """Apply rotary positional encoding. x: (B, n_heads, T, d_head)"""
    x1 = x[..., ::2]                                    # <1>
    x2 = x[..., 1::2]
    x_rot = torch.stack([-x2, x1], dim=-1).flatten(-2)  # <2>
    return x * cos + x_rot * sin


def make_rope_cache(max_seq_len, d_head, device):
    """Precompute cos/sin tables for RoPE."""
    theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    positions = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(positions, theta)               # (T, d_head//2)
    freqs = torch.cat([freqs, freqs], dim=-1)           # (T, d_head)
    cos = freqs.cos()[None, None, :, :]                 # (1, 1, T, d_head)
    sin = freqs.sin()[None, None, :, :]
    return cos, sin


class NanoMLA(nn.Module):
    """Multi-head Latent Attention (simplified, no decoupled RoPE).

    Standard MHA caches K and V directly: O(n_heads * d_head) per token.
    MLA compresses: cache only c_KV of dim d_compressed, then reconstruct K, V.
    KV cache reduction: 2*n_heads*d_head → d_compressed (typically 4x smaller).
    """

    def __init__(self, d_model: int, n_heads: int, d_compressed: int, max_seq_len: int = 1024):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.d_compressed = d_compressed

        self.W_c = nn.Linear(d_model, d_compressed, bias=False)  # compress to latent
        self.W_K = nn.Linear(d_compressed, d_model, bias=False)  # expand to keys
        self.W_V = nn.Linear(d_compressed, d_model, bias=False)  # expand to values
        self.W_Q = nn.Linear(d_model, d_model, bias=False)        # queries (not cached)
        self.W_O = nn.Linear(d_model, d_model, bias=False)        # output projection

        cos, sin = make_rope_cache(max_seq_len, self.d_head, device="cpu")
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x:    (B, T, d_model)
        mask: (1, 1, T, T) causal mask — True where attention is forbidden
        """
        B, T, _ = x.shape

        c_kv = self.W_c(x)                                      # (B, T, d_compressed)  # <1>
        K = self.W_K(c_kv)                                      # (B, T, d_model)        # <2>
        V = self.W_V(c_kv)
        Q = self.W_Q(x)                                         # (B, T, d_model)

        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        cos = self.cos[:, :, :T, :].to(x.device)
        sin = self.sin[:, :, :T, :].to(x.device)
        Q = apply_rope(Q, cos, sin)                             # <3>
        K = apply_rope(K, cos, sin)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        out = weights @ V

        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_O(out)


# Verify shapes
B, T, d = 2, 16, 384
mla = NanoMLA(d_model=d, n_heads=6, d_compressed=96)
x_in = torch.randn(B, T, d)
out = mla(x_in)
print(f"NanoMLA: input {tuple(x_in.shape)} → output {tuple(out.shape)}")
print(f"KV compressed dim: {mla.d_compressed} (vs full K+V: {2 * d} dims)")
print(f"Cache saving at inference: {2*mla.n_heads*mla.d_head / mla.d_compressed:.1f}x")

1. Compress input to low-rank latent $c_{KV}$ — this single vector per token is what gets stored in the KV cache at inference.
2. Expand $c_{KV}$ to full-rank K and V at attention time; at inference this reconstruction runs on the entire cached sequence.
3. Apply RoPE after reconstruction so position encoding does not need to be embedded in the cached latent.

## Mixture of Experts (MoE)

### The dense FFN problem

A standard FFN block computes:

$$\text{FFN}(x) = W_2 \cdot \text{SwiGLU}(W_1 x, W_3 x)$$

where $\text{SwiGLU}(a, b) = \text{SiLU}(a) \odot b$. All $P_{\text{FFN}}$ parameters activate for every token on every forward pass — total FLOPs per token scale linearly with parameter count, so more capacity always means proportionally more compute.

### The MoE solution

Replace the single FFN with $E$ expert FFNs. Per token, activate only the top-$k$ experts. DeepSeek uses two types:

- **Shared experts** ($N_s$): always active — handle universal patterns every token needs.
- **Routed experts** ($N_r$): conditionally active — each token activates its top-$k$ routed experts.

The routing decision is made by a linear router: given $x$, compute scores $s = x W_r \in \mathbb{R}^{N_r}$, take top-$k$, softmax over only those $k$ scores to get weights $g_i$, then:

$$\text{MoE}(x) = \sum_{i \in \text{shared}} E_i(x) + \sum_{i \in \text{top-}k} g_i \cdot E_i(x)$$

[Total params: $(N_s + N_r) \times P_{\text{expert}}$. Activated params per token: $(N_s + k) \times P_{\text{expert}}$.]{.mark} With $N_r = 55, k = 6$ (DeepSeek-V3), the sparsity ratio is approximately $10\times$.

### Expert load balancing

[Without regularization, the router collapses — all tokens route to the same few experts.]{.mark} The standard fix is an auxiliary load-balancing loss:

$$\mathcal{L}_{\text{aux}} = \alpha \cdot N_r \cdot \sum_{i=1}^{N_r} f_i \cdot P_i$$

where $f_i$ is the fraction of tokens routed to expert $i$ (non-differentiable, stops gradient), and $P_i$ is the average router probability for expert $i$ (differentiable). This penalizes routing imbalance: if one expert gets all tokens ($f_i = 1, f_j = 0$), the loss is large.

In [ ]:
class SwiGLU(nn.Module):
    """SwiGLU activation: SiLU(W1 x) * W3 x — used in LLaMA and DeepSeek."""

    def __init__(self, d_model: int, d_ffn: int):
        super().__init__()
        self.W1 = nn.Linear(d_model, d_ffn, bias=False)  # gate
        self.W3 = nn.Linear(d_model, d_ffn, bias=False)  # pass-through
        self.W2 = nn.Linear(d_ffn, d_model, bias=False)  # down-project

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.W2(F.silu(self.W1(x)) * self.W3(x))


class NanoMoE(nn.Module):
    """DeepSeek-style Mixture of Experts: shared + routed experts, top-k routing.

    Returns: (output, aux_loss) where aux_loss is the load-balancing penalty.
    """

    def __init__(
        self,
        d_model: int,
        d_ffn: int,
        n_shared: int = 1,
        n_routed: int = 8,
        top_k: int = 2,
        aux_loss_coeff: float = 1e-2,
    ):
        super().__init__()
        self.n_shared = n_shared
        self.n_routed = n_routed
        self.top_k = top_k
        self.aux_alpha = aux_loss_coeff

        self.shared_experts = nn.ModuleList([SwiGLU(d_model, d_ffn) for _ in range(n_shared)])
        self.routed_experts = nn.ModuleList([SwiGLU(d_model, d_ffn) for _ in range(n_routed)])
        self.router = nn.Linear(d_model, n_routed, bias=False)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x: (B, T, d_model)
        returns: output (B, T, d_model), aux_loss scalar
        """
        B, T, d = x.shape
        x_flat = x.view(B * T, d)

        # Shared experts: always active
        shared_out = sum(e(x_flat) for e in self.shared_experts)  # (B*T, d)  # <1>

        # Router: compute logits, select top-k experts
        logits = self.router(x_flat)                              # (B*T, n_routed)
        probs = F.softmax(logits, dim=-1)
        topk_vals, topk_idx = torch.topk(probs, self.top_k, dim=-1)
        gates = topk_vals / (topk_vals.sum(dim=-1, keepdim=True) + 1e-9)  # <2>

        # Routed expert computation: accumulate weighted outputs
        routed_out = torch.zeros_like(x_flat)
        for k_pos in range(self.top_k):
            expert_ids = topk_idx[:, k_pos]
            gate_vals = gates[:, k_pos].unsqueeze(-1)
            for expert_id in range(self.n_routed):
                mask = expert_ids == expert_id
                if mask.any():
                    routed_out[mask] += gate_vals[mask] * self.routed_experts[expert_id](x_flat[mask])

        output = (shared_out + routed_out).view(B, T, d)

        # Auxiliary load-balancing loss
        with torch.no_grad():
            top1_idx = topk_idx[:, 0]
            one_hot = F.one_hot(top1_idx, self.n_routed).float()
            f = one_hot.mean(dim=0)                               # fraction per expert  # <3>
        P = probs.mean(dim=0)                                     # avg probability per expert
        aux_loss = self.aux_alpha * self.n_routed * (f * P).sum()

        return output, aux_loss


# Verify shapes and aux loss
moe = NanoMoE(d_model=384, d_ffn=384 * 4, n_shared=1, n_routed=8, top_k=2)
x_in = torch.randn(2, 16, 384)
out, aux = moe(x_in)
print(f"NanoMoE: input {tuple(x_in.shape)} → output {tuple(out.shape)}")
print(f"aux_loss: {aux.item():.4f}")
print(f"Activated experts per token: {moe.n_shared + moe.top_k} / {moe.n_shared + moe.n_routed} total")

1. Shared experts run unconditionally on every token — they learn universal, token-agnostic transformations.
2. Re-normalize the top-$k$ gate weights over the selected subset so they sum to 1; this stabilizes gradient magnitude.
3. $f_i$ uses `torch.no_grad()` because it is a hard assignment indicator — its gradient is zero everywhere, so backprop must flow through $P_i$ alone.

## Supporting Components: RMSNorm

DeepSeek uses **RMSNorm** instead of LayerNorm. The difference: RMSNorm skips the mean-centering step, normalizing by the RMS of activations only:

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \qquad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2 + \epsilon}$$

[This is slightly faster (no mean subtraction) and performs equally well in practice.]{.underline} Removing one reduction kernel saves approximately 15% compute per normalization call, and avoids the numerical instability that can occur when the mean is non-zero and large in very deep networks.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return (x / rms) * self.gamma


# Verify RMSNorm is approximately unit-RMS post-norm
rn = RMSNorm(384)
x = torch.randn(4, 16, 384) * 5.0   # large activations
y = rn(x)
print(f"Input  RMS: {x.pow(2).mean().sqrt().item():.3f}")
print(f"Output RMS: {y.pow(2).mean().sqrt().item():.3f}  (≈1.0 with unit gamma init)")

## Assembling NanoDeepSeek

We now have all the pieces. The `NanoDeepSeek` block follows DeepSeek's pre-norm residual pattern:

```
x  →  RMSNorm → MLA → + x  →  RMSNorm → MoE → + x
```

The full model stacks $L$ such blocks between token embeddings and the LM head, with weight tying between the embedding matrix and the LM head projection.

In [ ]:
@dataclass
class NanoDeepSeekConfig:
    vocab_size:     int   = 50257
    d_model:        int   = 384
    n_layers:       int   = 6
    n_heads:        int   = 6
    d_compressed:   int   = 96     # MLA KV latent dim  (d_model // 4)
    d_ffn:          int   = 1024   # per-expert hidden dim
    n_shared:       int   = 1      # shared experts (always active)
    n_routed:       int   = 8      # routed expert pool size
    top_k:          int   = 2      # routed experts activated per token
    max_seq_len:    int   = 256
    aux_loss_coeff: float = 1e-2


def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    return torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device),
        diagonal=1,
    ).unsqueeze(0).unsqueeze(0)   # (1, 1, T, T)


class DeepSeekBlock(nn.Module):
    """One transformer block: pre-norm MLA + pre-norm MoE."""

    def __init__(self, config: NanoDeepSeekConfig):
        super().__init__()
        self.norm_attn = RMSNorm(config.d_model)
        self.attn = NanoMLA(
            d_model=config.d_model,
            n_heads=config.n_heads,
            d_compressed=config.d_compressed,
            max_seq_len=config.max_seq_len,
        )
        self.norm_ffn = RMSNorm(config.d_model)
        self.moe = NanoMoE(
            d_model=config.d_model,
            d_ffn=config.d_ffn,
            n_shared=config.n_shared,
            n_routed=config.n_routed,
            top_k=config.top_k,
            aux_loss_coeff=config.aux_loss_coeff,
        )

    def forward(
        self, x: torch.Tensor, mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        x = x + self.attn(self.norm_attn(x), mask)
        moe_out, aux_loss = self.moe(self.norm_ffn(x))
        x = x + moe_out
        return x, aux_loss


class NanoDeepSeek(nn.Module):
    """Nano DeepSeek: MLA + MoE transformer for the capstone series."""

    def __init__(self, config: NanoDeepSeekConfig):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.blocks = nn.ModuleList([DeepSeekBlock(config) for _ in range(config.n_layers)])
        self.norm_out = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight  # weight tying

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        idx:     (B, T) token indices
        targets: (B, T) next-token targets, or None
        returns: logits (B, T, vocab), total_loss (lm_loss + sum aux_losses) or None
        """
        B, T = idx.shape
        x = self.token_embedding(idx)           # (B, T, d_model)
        mask = make_causal_mask(T, idx.device)

        total_aux = torch.tensor(0.0, device=idx.device)
        for block in self.blocks:
            x, aux = block(x, mask)
            total_aux = total_aux + aux

        x = self.norm_out(x)
        logits = self.lm_head(x)                # (B, T, vocab_size)

        loss = None
        if targets is not None:
            lm_loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )
            loss = lm_loss + total_aux

        return logits, loss


# Quick forward pass check
cfg = NanoDeepSeekConfig()
model = NanoDeepSeek(cfg)
idx = torch.randint(0, cfg.vocab_size, (2, 16))
targets = torch.randint(0, cfg.vocab_size, (2, 16))
logits, loss = model(idx, targets)
print(f"NanoDeepSeek forward pass OK")
print(f"  logits: {tuple(logits.shape)}")
print(f"  loss:   {loss.item():.4f}")

The total loss passed to the optimizer is the sum of the language modeling cross-entropy and the per-layer auxiliary load-balancing losses. This ensures the router is trained jointly with the model parameters.

:::{.callout-caution}
The `NanoMoE.forward` loop iterates over all expert IDs for every top-$k$ slot, giving $O(k \times N_r)$ conditional dispatches per batch. This is pedagogically clear but inefficient at scale. Production implementations use scatter/gather kernels (e.g. `torch.ops.moe_dispatch`) that batch tokens by expert assignment in a single pass.

:::

## Parameter Count and KV Cache Comparison

We now compare NanoGPT against NanoDeepSeek. The key distinction is between *total* and *activated* parameters — MoE decouples these two quantities.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


def count_activated_params_ds(config: NanoDeepSeekConfig) -> int:
    """Estimate activated parameters per forward token (excluding embedding)."""
    d, d_c = config.d_model, config.d_compressed

    # MLA: W_c + W_K + W_V + W_Q + W_O
    p_mla = (d * d_c) + (d_c * d) + (d_c * d) + (d * d) + (d * d)

    # MoE: (n_shared + top_k) active experts each with W1 + W3 + W2
    p_expert = 3 * config.d_model * config.d_ffn
    p_moe_active = (config.n_shared + config.top_k) * p_expert

    return config.n_layers * (p_mla + p_moe_active)


cfg_ds = NanoDeepSeekConfig()
model_ds = NanoDeepSeek(cfg_ds)
total_ds = count_params(model_ds)
active_ds = count_activated_params_ds(cfg_ds)

# NanoGPT reference values (from Notebook 01, SwiGLU FFN, ~29.9M total)
nanogpt_total = 29.9e6
nanogpt_active = 29.9e6  # dense: all params activate

def kv_cache_MB(n_heads, d_head, n_layers, seq_len, bpe=2):
    return 2 * n_heads * d_head * n_layers * seq_len * bpe / 1e6

def kv_cache_MB_mla(d_c, n_layers, seq_len, bpe=2):
    return d_c * n_layers * seq_len * bpe / 1e6

seq = 256

print("=" * 62)
print(f"{'':32s} {'NanoGPT':>12s} {'NanoDeepSeek':>14s}")
print("=" * 62)
print(f"{'Total parameters':32s} {'~29.9M':>12s} {total_ds/1e6:>13.1f}M")
print(f"{'Activated per token':32s} {'~29.9M':>12s} {active_ds/1e6:>13.1f}M")
print(f"{'KV cache @ seq=256 (MB)':32s} {kv_cache_MB(6,64,6,seq):>11.2f}M {kv_cache_MB_mla(cfg_ds.d_compressed,6,seq):>13.2f}M")
print(f"{'KV cache @ seq=1024 (MB)':32s} {kv_cache_MB(6,64,6,1024):>11.2f}M {kv_cache_MB_mla(cfg_ds.d_compressed,6,1024):>13.2f}M")
print(f"{'Attention':32s} {'MHA (RoPE)':>12s} {'MLA (low-rank KV)':>14s}")
print(f"{'FFN':32s} {'Dense SwiGLU':>12s} {'MoE (1s+8r, top-2)':>14s}")
print(f"{'Normalisation':32s} {'RMSNorm':>12s} {'RMSNorm':>14s}")
print("=" * 62)

The table shows the key tradeoff: MoE gives NanoDeepSeek more total parameters (larger capacity for the same training compute budget) while the activated parameter count per token is significantly lower. MLA reduces KV cache by ~4x at the nano scale, with the ratio growing at larger model sizes where `d_model ≫ d_compressed`.

## Memory Taxonomy — Three Axes of Sparsity

It is useful to place DeepSeek's architecture in the broader context of memory types a transformer can leverage:

| Memory type | Update speed | Persistence | Access mechanism |
|---|---|---|---|
| Dense parameters | Slow (gradient descent) | Permanent across all inputs | Dense matmul, all tokens |
| KV cache | Fast | Per-sequence | Attention — content-based lookup |
| MoE experts | Slow (gradient descent) | Permanent | Top-$k$ routing, token-conditional |
| Engram (Notebook 03) | Fast | Permanent | $O(1)$ hash lookup, $n$-gram conditional |

: {tbl-colwidths="[22,20,20,38]"}

MLA reduces the *cost* of the KV cache. MoE conditions which FFN parameters activate on a per-token basis. [The Engram layer (Notebook 03) introduces a fourth axis: $n$-gram conditional lookup into a static embedding table — no attention, no routing, just a hash.]{.underline}

## Summary

| Component | What it does | Key parameter |
|---|---|---|
| `NanoMLA` | Low-rank KV compression: cache $d_c$ instead of $n_h \cdot d_h$ | `d_compressed` |
| Decoupled RoPE | Separates position encoding from compressed latent | $d_r$ RoPE slice |
| `NanoMoE` | Top-$k$ routing over $N_r$ routed + $N_s$ shared experts | `n_routed`, `top_k` |
| Aux loss | Penalizes routing imbalance: $\alpha N_r \sum f_i P_i$ | `aux_loss_coeff` |
| `RMSNorm` | Normalize by RMS, skip mean shift | `eps` |
| `SwiGLU` | $W_2(\text{SiLU}(W_1 x) \odot W_3 x)$ — used in every expert | `d_ffn` |
| `NanoDeepSeek` | Full model: embedding → `DeepSeekBlock` × $L$ → RMSNorm → LM head | `NanoDeepSeekConfig` |

: {tbl-colwidths="[18,52,30]"}

**Notebook 02** trains NanoGPT and NanoDeepSeek side-by-side on FineWeb-Edu at matched activated parameter count, comparing loss curves and measuring whether MLA + MoE improves loss per activated FLOP.

## Exercises

1. **Verify compression ratio.** For a model with `d_model=1024`, `n_heads=16`, `d_compressed=128`, compute the KV cache size in MB for sequence lengths 512, 2048, and 8192. By what factor does MLA reduce the cache compared to standard MHA?

2. **Decoupled RoPE.** The current `NanoMLA` applies RoPE after reconstructing K from $c_{KV}$. Implement the decoupled variant: add a separate `W_KR` projection from the original `x` that produces a `d_rope`-dimensional key slice, cache `c_kv` and `k_r`, and concatenate `[K_content, K_rope]` before the dot product.

3. **Router collapse.** Train `NanoMoE` with `aux_loss_coeff=0.0` for 500 steps on random data and plot the fraction of tokens routed to each expert. Then repeat with `aux_loss_coeff=1e-2`. How does the load distribution differ?

4. **Expert utilization.** Modify `NanoMoE.forward` to return, alongside `aux_loss`, a tensor `expert_load` of shape `(n_routed,)` giving the fraction of batch tokens assigned to each expert. Log this during training and plot it over time.

5. **Capacity factor.** Production MoE implementations drop tokens when an expert exceeds its capacity budget. Add a `capacity_factor` parameter to `NanoMoE`: if more than `capacity_factor * (B*T / n_routed)` tokens are routed to any expert, drop the excess tokens (set their contribution to zero). Verify the model still trains without NaN.

6. **Activated vs total FLOPs.** Extend `count_activated_params_ds` to also estimate the number of multiply-add operations per token for `NanoDeepSeek`, and compare with `NanoGPT` at matched `d_model`. At what sparsity ratio (top-$k$ / $N_r$) does NanoDeepSeek achieve equal FLOPs per token to NanoGPT?

:::{.callout-note}
## References
- DeepSeek-AI (2024). *DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model.* arXiv:2405.04434.
- Dai et al. (2024). *DeepSeekMoE: Towards Ultimate Expert Specialization in Mixture-of-Experts Language Models.* arXiv:2401.06066.
- Zhang & Sennrich (2019). *Root Mean Square Layer Normalization.* NeurIPS.

:::

■